# Cloudflare AI Search

This notebook covers how to get started with Cloudflare AI Search instance administration and retrieval.

## Setup

This Python package wraps Cloudflare's REST API. To interact with AI Search, provide an API token with the appropriate privileges.

You can create and manage API tokens here:

https://dash.cloudflare.com/YOUR-ACCT-NUMBER/api-tokens

### Credentials

For this notebook, use a token with **AI Search:Edit** and **AI Search:Run** permissions.

You can use `CF_AI_SEARCH_API_TOKEN` for AI Search-specific access or `CF_API_TOKEN` if you have a broader token. The examples also read `CF_ACCOUNT_ID` and optionally `CF_AI_SEARCH_NAMESPACE` from the environment.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(".env")

# Declare as variables for example's sake (you'll see why below)
cf_acct_id = os.getenv("CF_ACCOUNT_ID")

# AI Search token with AI Search:Edit and AI Search:Run permissions
cf_ai_search_token = os.getenv("CF_AI_SEARCH_API_TOKEN")

# OR, a single broader Cloudflare token
api_token = os.getenv("CF_API_TOKEN")

# AI Search instances live in a namespace. The default namespace is usually enough.
ai_search_namespace = os.getenv("CF_AI_SEARCH_NAMESPACE", "default")

token = cf_ai_search_token or api_token

## Initialization

In [ ]:
import uuid
import warnings

from langchain_cloudflare import (
    CloudflareAISearchClient,
    CloudflareAISearchRetriever,
)

warnings.filterwarnings("ignore")

In [ ]:
# name your AI Search instance
ai_search_instance_name = f"test-langchain-ai-search-{uuid.uuid4().hex[:8]}"

# use a unique phrase so we can search for exactly the content we upload
query_marker = f"langchain-ai-search-notebook-{uuid.uuid4().hex}"

### CloudflareAISearchClient Class

Now we can create the `CloudflareAISearchClient` instance. Here we passed:

* The account ID
* An AI Search token or broader Cloudflare API token
* The AI Search namespace

In [ ]:
client = CloudflareAISearchClient(
    account_id=cf_acct_id,
    api_token=token,
    namespace=ai_search_namespace,
)

### Cleanup

Before we get started, let's delete any `test-langchain-ai-search-*` instances left over from earlier notebook runs.

In [ ]:
arr_instances = client.list_instances(search="test-langchain-ai-search")
arr_instances = [
    x for x in arr_instances if x.get("id", "").startswith("test-langchain-ai-search-")
]

for instance in arr_instances:
    client.delete_instance(instance.get("id"), missing_ok=True)

print(f"Deleted {len(arr_instances)} old notebook instances")

## Manage AI Search

### Creating an Instance

Let's start by creating a temporary AI Search instance. New AI Search instances include built-in storage, so we can upload files directly to the instance.

In [ ]:
instance = client.create_instance(ai_search_instance_name)
print(instance)

### Listing Instances

Now, we can list AI Search instances in the namespace on our account.

In [ ]:
instances = client.list_instances(search=ai_search_instance_name)
[x.get("id") for x in instances]

### Instance Info and Stats

We can retrieve instance configuration and indexing stats.

In [ ]:
client.get_instance(ai_search_instance_name)

In [ ]:
client.stats(ai_search_instance_name)

### Uploading Items

AI Search can index files uploaded to built-in storage. This example uploads one small Markdown document and waits for indexing to complete.

In [ ]:
content = "\n".join(
    [
        "# LangChain Cloudflare AI Search notebook",
        "",
        f"{query_marker} validates AI Search notebook retrieval.",
        "Cloudflare AI Search indexes uploaded files for natural language search.",
    ]
)

item = client.upload_item(
    "notebook-guide.md",
    content,
    instance_name=ai_search_instance_name,
    content_type="text/markdown",
    metadata={"source": "notebook"},
    wait_for_completion=True,
)

if item.get("status") != "completed":
    item = client.wait_for_item(item["id"], instance_name=ai_search_instance_name)

item

### Listing Items

Now, we can inspect uploaded items for this instance.

In [ ]:
items = client.list_items(ai_search_instance_name)
[(x.get("id"), x.get("key"), x.get("status")) for x in items]

## Query AI Search

We can run a raw AI Search query directly through the admin client. The result contains chunks from the indexed files.

In [ ]:
results = client.search(
    query_marker,
    instance_name=ai_search_instance_name,
    ai_search_options={
        "retrieval": {"max_num_results": 3},
        "query_rewrite": {"enabled": False},
        "reranking": {"enabled": False},
    },
)

[(chunk.get("text"), chunk.get("score")) for chunk in results.get("chunks", [])]

### Query by Turning into Retriever

You can also use the AI Search instance as a LangChain retriever for easier usage in chains and agents.

In [ ]:
retriever = CloudflareAISearchRetriever(
    account_id=cf_acct_id,
    api_token=token,
    namespace=ai_search_namespace,
    instance_name=ai_search_instance_name,
    k=3,
    rewrite_query=False,
    reranking=False,
)

docs = retriever.invoke(query_marker)
[(doc.page_content, doc.metadata) for doc in docs]

## Async Examples

The admin client also exposes async methods for REST usage and Python Worker bindings. In a Python Worker, pass an `ai_search_namespaces` binding instead of REST credentials:

```python
client = CloudflareAISearchClient(binding=env.AI_SEARCH)
instance = await client.acreate_instance("tenant-a")
await client.adelete_instance(instance["id"])
```

## Cleanup

Let's finish by deleting the item and AI Search instance we created in this notebook.

In [ ]:
if item.get("id"):
    client.delete_item(
        item["id"],
        instance_name=ai_search_instance_name,
        missing_ok=True,
    )

client.delete_instance(ai_search_instance_name, missing_ok=True)

## API Reference

For more information, see:

* [Cloudflare AI Search docs](https://developers.cloudflare.com/ai-search/)
* [AI Search REST API](https://developers.cloudflare.com/api/resources/ai_search/)
* [AI Search Workers binding](https://developers.cloudflare.com/ai-search/api/instances/workers-binding/)